## Модули


In [53]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

## Получение данных


Pclass - класс билета, социально-экономический статус пассажира:
* 1 = Первый класс (Высший)
* 2 = Второй класс (Средний)
* 3 = Третий класс (Низший)

SibSp — Количество родственников на борту (Siblings/Spouses). Сумма:
* Братьев и сестер (sibling)
* Мужей и жен (spouse)

Parch — Количество родственников на борту (Parents/Children). Сумма:
* Родителей (parent)
* Детей (child)

Embarked — Порт посадки. Буквенный код:
* C = Cherbourg (Шербур, Франция)
* Q = Queenstown (Куинстаун, ныне Коб, Ирландия)
* S = Southampton (Саутгемптон, Великобритания)

Fare — Стоимость билета. Плата за проезд в британских фунтах.


In [ ]:
base_features = [
    "PassengerId",
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "Survived",
]

train_df = pd.read_csv("./titanic_train.csv")[base_features]
print(train_df.head())

test_df = pd.read_csv("./titanic_test.csv")[base_features[:-1]].merge(
    right=pd.read_csv("./gender_submission.csv"),
    how="left",
    on="PassengerId",
)
print(test_df.head())

   PassengerId  Pclass     Sex   Age  SibSp  Parch     Fare Embarked  Survived
0            1       3    male  22.0      1      0   7.2500        S         0
1            2       1  female  38.0      1      0  71.2833        C         1
2            3       3  female  26.0      0      0   7.9250        S         1
3            4       1  female  35.0      1      0  53.1000        S         1
4            5       3    male  35.0      0      0   8.0500        S         0
   PassengerId  Pclass     Sex   Age  SibSp  Parch     Fare Embarked  Survived
0          892       3    male  34.5      0      0   7.8292        Q         0
1          893       3  female  47.0      1      0   7.0000        S         1
2          894       2    male  62.0      0      0   9.6875        Q         0
3          895       3    male  27.0      0      0   8.6625        S         0
4          896       3  female  22.0      1      1  12.2875        S         1


In [46]:
def update_df(df: pd.DataFrame) -> pd.DataFrame:
    updated_df = df.copy()

    le_sex = LabelEncoder()
    updated_df["Sex"] = le_sex.fit_transform(df["Sex"])

    le_embarked = LabelEncoder()
    updated_df["Embarked"] = le_embarked.fit_transform(df["Embarked"])

    updated_df["Family"] = df["SibSp"] + df["Parch"] + 1

    updated_df["Fare"] = df.groupby("Pclass")["Fare"].transform(
        lambda x: x.fillna(x.median())
    )

    updated_df["Age"] = df.groupby(["Pclass", "Sex"])["Age"].transform(
        lambda x: x.fillna(x.median())
    )

    updated_features = [
        "Pclass",
        "Sex",
        "Age",
        "Family",
        "Embarked",
        "Fare",
        "Survived",
    ]

    updated_df = updated_df[updated_features]

    return updated_df


cleared_train_df = update_df(train_df)
print(cleared_train_df.head())

train_X, train_y = cleared_train_df.iloc[:, :-1], cleared_train_df["Survived"]
print(train_X, train_y, sep="\n")

cleared_test_df = update_df(test_df)
print(cleared_test_df.head())

train_X, train_y = cleared_test_df.iloc[:, :-1], cleared_test_df["Survived"]
print(train_X, train_y, sep="\n")

   Pclass  Sex   Age  Family  Embarked     Fare  Survived
0       3    1  22.0       2         2   7.2500         0
1       1    0  38.0       2         0  71.2833         1
2       3    0  26.0       1         2   7.9250         1
3       1    0  35.0       2         2  53.1000         1
4       3    1  35.0       1         2   8.0500         0
     Pclass  Sex   Age  Family  Embarked     Fare
0         3    1  22.0       2         2   7.2500
1         1    0  38.0       2         0  71.2833
2         3    0  26.0       1         2   7.9250
3         1    0  35.0       2         2  53.1000
4         3    1  35.0       1         2   8.0500
..      ...  ...   ...     ...       ...      ...
886       2    1  27.0       1         2  13.0000
887       1    0  19.0       1         2  30.0000
888       3    0  21.5       4         2  23.4500
889       1    1  26.0       1         0  30.0000
890       3    1  32.0       1         1   7.7500

[891 rows x 6 columns]
0      0
1      1
2      1
3

In [48]:
X_train, X_val, y_train, y_val = train_test_split(
    train_X,
    train_y,
    test_size=0.2,
    random_state=42,
    stratify=train_y,
)

print(f"Размеры данных:")
print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_val: {y_val.shape}")

Размеры данных:
X_train: (334, 6)
X_val: (84, 6)
y_train: (334,)
y_val: (84,)


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,  # количество деревьев
    max_depth=5,  # максимальная глубина
    random_state=42,  # для воспроизводимости
    min_samples_split=10,  # минимальное количество образцов для разделения узла
    min_samples_leaf=4,  # минимальное количество образцов в листе
)

# Обучение модели
rf_model.fit(X_train, y_train)

# Предсказания на валидационной выборке
y_pred = rf_model.predict(X_val)
y_pred_proba = rf_model.predict_proba(X_val)[:, 1]  # вероятности для класса 1

# Оценка модели
accuracy = accuracy_score(y_val, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred))

Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        53
           1       1.00      1.00      1.00        31

    accuracy                           1.00        84
   macro avg       1.00      1.00      1.00        84
weighted avg       1.00      1.00      1.00        84

